In [26]:
import pandas as pd
import sys
import os
from sklearn import set_config
import numpy as np
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import train_test_split
from typing import Tuple
set_config(display="text")  # displays text representation of estimators

from sksurv.util import Surv
sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
from src.utils.cox_models import Cox_regression, p_values_Cox_regression
%load_ext autoreload
%autoreload 2
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
pp = Preprocessor()
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_keep = pp.clean_columns_dataset(df_clinical_data)



In [3]:
list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df
df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
df_rsem_z_scores = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt")
clean_df = pp.eliminate_nan_genes(df_rsem_z_scores, "Hugo_Symbol")



Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 298 - Total(%) 0.36
Shape of the CSV: (20440, 819)
Shape of the CSV: (20440, 819)


In [4]:

"""expression = df_ESR1_merged["expression"].values
lsit = sorted(expression)
plt.plot((lsit)) # plotting by columns
plt.show()
"""

'expression = df_ESR1_merged["expression"].values\nlsit = sorted(expression)\nplt.plot((lsit)) # plotting by columns\nplt.show()\n'

In [5]:

"""
ESR1 = df_ESR1_merged["expression"].values.reshape(-1,1)

scaler = StandardScaler()
z_score = scaler.fit_transform(ESR1)

plt.figure(figsize=(12,5))
plt.title("box plot of the z score")
plt.boxplot(z_score)
plt.show()

plt.figure(figsize=(12,5))
plt.hist(z_score, bins=30)
plt.title("Distribution of the z score")
plt.xlabel("z-score")
plt.ylabel("Frecuencia")
plt.show()
print(f"Min number ESR1: {ESR1.min()}")
print(f"Max number ESR1: {ESR1.max()}")
#plt.xlabel(f"Max number in ESR1:{ESR1.max()} ")"""

'\nESR1 = df_ESR1_merged["expression"].values.reshape(-1,1)\n\nscaler = StandardScaler()\nz_score = scaler.fit_transform(ESR1)\n\nplt.figure(figsize=(12,5))\nplt.title("box plot of the z score")\nplt.boxplot(z_score)\nplt.show()\n\nplt.figure(figsize=(12,5))\nplt.hist(z_score, bins=30)\nplt.title("Distribution of the z score")\nplt.xlabel("z-score")\nplt.ylabel("Frecuencia")\nplt.show()\nprint(f"Min number ESR1: {ESR1.min()}")\nprint(f"Max number ESR1: {ESR1.max()}")\n#plt.xlabel(f"Max number in ESR1:{ESR1.max()} ")'

In [7]:
def split_data_set(df_mRNA : pd.DataFrame, 
                       df_clinical : pd.DataFrame,
                       gene : str) -> Tuple:
    
        df_single_gene  = pp.gene_to_long(df_mRNA, gene)
        
        df_gene_merged = df_single_gene.merge(df_clinical, on="Sample ID", how="inner")
        
        df_gene_merged["Overall Survival (Months)"] = pd.to_numeric(df_gene_merged["Overall Survival (Months)"])
        
        status = df_gene_merged["Overall Survival Status"].astype(str).str.strip()
        
        df_gene_merged["event"] = status.str.contains("DECEASED", na=False) 
    
        df_gene_merged = df_gene_merged.dropna(subset=["Overall Survival (Months)"])
        
        df_gene_merged["expression"] = pd.to_numeric(df_gene_merged["expression"], errors="coerce")
        
        X = df_gene_merged[["expression"]]
        
        Y_surv = Surv.from_dataframe(
        event="event",
        time="Overall Survival (Months)",
        data=df_gene_merged
        )
        
        return X, Y_surv, df_gene_merged

In [30]:
X_MKI67, Y_surv_MKI67, df_gene_merged_MKI67 = split_data_set(clean_df,df_clinical_keep, "BCL2")
X_train_MKI67, X_test_MKI67, Y_train_MKI67, Y_test_MKI67 = train_test_split(
    X_MKI67, Y_surv_MKI67, train_size=0.80, test_size=0.20, random_state=42
)
betas_MKI67, chp_predict_MKI673, survival_curve_MKI67, risk_curve_MKI67 = Cox_regression(X_train_MKI67, Y_train_MKI67, X_test_MKI67)

df_life_MKI67 = df_gene_merged_MKI67[["expression", "event", "Overall Survival (Months)"]]
df_life_MKI67 = df_life_MKI67.dropna(subset="Overall Survival (Months)")
df_life_MKI67 = df_life_MKI67.dropna(subset="expression")
thr = df_life_MKI67["expression"].median()
df_life_MKI67["group_bin"] = (df_life_MKI67["expression"] >= thr).astype(int)


value = p_values_Cox_regression(df_life_MKI67,event_col="event", duration_col="Overall Survival (Months)")
value["p"]


covariate
expression    0.460776
group_bin     0.351281
Name: p, dtype: float64

In [ ]:
df_life_BCL2 = df_gene_merged_MKI67[["expression", "event", "Overall Survival (Months)"]].copy()

df_life_BCL2["expression"] = pd.to_numeric(df_life_BCL2["expression"], errors="coerce")
df_life_BCL2["Overall Survival (Months)"] = pd.to_numeric(df_life_BCL2["Overall Survival (Months)"], errors="coerce")

df_life_BCL2 = df_life_BCL2.dropna(subset=["expression", "event", "Overall Survival (Months)"])


df_life_BCL2["time_60"] = np.minimum(df_life_BCL2["Overall Survival (Months)"], 60)

df_life_BCL2["event_60"] = df_life_BCL2["event"].copy()

df_life_BCL2.loc[df_life_BCL2["Overall Survival (Months)"] > 60, "event_60"] = False

df_cox_60 = df_life_BCL2[["expression", "time_60", "event_60"]].copy()

value_60 = p_values_Cox_regression(
    df_cox_60,
    event_col="event_60",
    duration_col="time_60"
)

print(value_60["p"].item())

0.00011236727834182663


In [49]:
genes_expression = [
    "TBC1D9",
    "SUSD3",
    "SLC39A6",
    "GFRA1",
    "SOX11",
    "GATA3",
    "SLC15A2",
    "NANOS1",
    "ZNF552",
    "ESR1",
    "NAT1",
    "NME3",
    "DNALI1",
    "AGR3",
    "CA12",
    "BCL2",
    "MKI67",
    "TP53",
    "ESR1",
    "AURKA"
]
genes_significant_p_value = []
genes_non_significant_p_value = []

for gene_sample in genes_expression:
    X, Y_surv, df_gene_merged = split_data_set(clean_df,df_clinical_keep, gene_sample)

    df_life = df_gene_merged[["expression", "event", "Overall Survival (Months)"]].copy()
    
    df_life["expression"] = pd.to_numeric(df_life["expression"], errors="coerce")

    df_life["Overall Survival (Months)"] = pd.to_numeric(df_life["Overall Survival (Months)"], errors="coerce").astype(float)
    
    df_life = df_life.dropna(subset=["expression", "event", "Overall Survival (Months)"])
    
    df_life["time_60"] = np.minimum(df_life["Overall Survival (Months)"], 60)
    
    df_life["event_60"] = df_life["event"].copy()
    
    
    df_life.loc[df_life["Overall Survival (Months)"] > 60,  "event_60"] = False
    
    df_cox_60_months = df_life[["expression", "time_60", "event_60"]]
    
    value = p_values_Cox_regression(df_cox_60_months,event_col="event_60", duration_col="time_60")
    
    p = value["p"].item()
    if p <= 0.05:
        genes_significant_p_value.append(f"Gene: {gene_sample} and the P-value: {p}")
    else:
        genes_non_significant_p_value.append(f"Gene: {gene_sample} and the P-value: {p}")

In [50]:
genes_non_significant_p_value

['Gene: SLC15A2 and the P-value: 0.052991520556734464',
 'Gene: ESR1 and the P-value: 0.11598917228131136',
 'Gene: NAT1 and the P-value: 0.0795026834666013',
 'Gene: NME3 and the P-value: 0.10397746981886452',
 'Gene: TP53 and the P-value: 0.412716192091657',
 'Gene: ESR1 and the P-value: 0.11598917228131136']

In [51]:
genes_significant_p_value

['Gene: TBC1D9 and the P-value: 0.033114249093091366',
 'Gene: SUSD3 and the P-value: 0.0001318866260015989',
 'Gene: SLC39A6 and the P-value: 0.004457263204402282',
 'Gene: GFRA1 and the P-value: 0.0030194448135109925',
 'Gene: SOX11 and the P-value: 0.018557912034319026',
 'Gene: GATA3 and the P-value: 0.01947459933616335',
 'Gene: NANOS1 and the P-value: 0.044759952227313306',
 'Gene: ZNF552 and the P-value: 0.0035767037946882467',
 'Gene: DNALI1 and the P-value: 0.015135559714935024',
 'Gene: AGR3 and the P-value: 0.010853077712437518',
 'Gene: CA12 and the P-value: 0.03398696207291381',
 'Gene: BCL2 and the P-value: 0.00011236727834182663',
 'Gene: MKI67 and the P-value: 0.03631303498614305',
 'Gene: AURKA and the P-value: 0.0418605650981712']

In [11]:
from lifelines.statistics import logrank_test

results_list = []

for gene_sample in genes_expression:
    df_gene = pp.gene_to_long(clean_df, gene_sample)
    df_gene_merged = df_gene.merge(df_clinical_keep, on="Sample ID", how="inner")

    df_gene_merged["expression"] = pd.to_numeric(df_gene_merged["expression"], errors="coerce")
    df_gene_merged["Overall Survival (Months)"] = pd.to_numeric(
        df_gene_merged["Overall Survival (Months)"], errors="coerce"
    )

    status = df_gene_merged["Overall Survival Status"].astype(str).str.strip()
    df_gene_merged["event"] = status.str.contains("DECEASED", na=False)

    df_gene_merged = df_gene_merged.dropna(
        subset=["expression", "Overall Survival (Months)", "event"]
    ).copy()

    thr = df_gene_merged["expression"].median()

    low_group = df_gene_merged[df_gene_merged["expression"] < thr]
    high_group = df_gene_merged[df_gene_merged["expression"] >= thr]

    results = logrank_test(
        durations_A=low_group["Overall Survival (Months)"],
        durations_B=high_group["Overall Survival (Months)"],
        event_observed_A=low_group["event"],
        event_observed_B=high_group["event"]
    )

    results_list.append({
        "gene": gene_sample,
        "p_value": results.p_value,
        "significant": results.p_value <= 0.05
    })

df_results = pd.DataFrame(results_list)
print(df_results)

       gene   p_value  significant
0    TBC1D9  0.109986        False
1     SUSD3  0.005156         True
2   SLC39A6  0.142706        False
3     GFRA1  0.062568        False
4     SOX11  0.062218        False
5     GATA3  0.237141        False
6   SLC15A2  0.672886        False
7    NANOS1  0.142905        False
8    ZNF552  0.025126         True
9      ESR1  0.689058        False
10     NAT1  0.487759        False
11     NME3  0.601688        False
12   DNALI1  0.050060        False
13     AGR3  0.591130        False
14     CA12  0.381673        False
15     BCL2  0.016147         True
16    MKI67  0.238531        False
17     TP53  0.023690         True
18     ESR1  0.689058        False
19    AURKA  0.247520        False


['Gene: SUSD3 and the P-value: 0.6464962285747877',
 'Gene: SLC39A6 and the P-value: 0.4776942274995962',
 'Gene: GFRA1 and the P-value: 0.26187068942289227',
 'Gene: SOX11 and the P-value: 0.19942305909361924',
 'Gene: GATA3 and the P-value: 0.6101658572062589',
 'Gene: SLC15A2 and the P-value: 0.8911498353425168',
 'Gene: NANOS1 and the P-value: 0.6874202798248465',
 'Gene: ZNF552 and the P-value: 0.06013947609505957',
 'Gene: ESR1 and the P-value: 0.5389137987929216',
 'Gene: NAT1 and the P-value: 0.520771070512495',
 'Gene: NME3 and the P-value: 0.4630160240807195',
 'Gene: DNALI1 and the P-value: 0.05862790250489832',
 'Gene: AGR3 and the P-value: 0.9688955481136952',
 'Gene: CA12 and the P-value: 0.5525076073382711',
 'Gene: BCL2 and the P-value: 0.35128078717795475',
 'Gene: MKI67 and the P-value: 0.5001839174529933',
 'Gene: TP53 and the P-value: 0.1124485555786122',
 'Gene: ESR1 and the P-value: 0.5389137987929216',
 'Gene: AURKA and the P-value: 0.5970327730729998']

['Gene: TBC1D9 and the P-value: 0.04532713888131658']